# 00 — Deepnote Setup, Preflight, and Excluded P0 Technical Pilot

Technical validation only before confirmatory collection. This notebook installs/checks dependencies, loads the shared experiment module, checks all seven model APIs, checks numerical measurement parsing, and runs an excluded P0 end-to-end technical pilot.


## 1. Install packages


In [ ]:
%pip install -q openai anthropic google-genai pandas numpy scipy statsmodels matplotlib tqdm python-dotenv


## 2. Load the shared experiment module


In [ ]:
import os, sys, json, asyncio, importlib.util
from pathlib import Path
import pandas as pd

candidates = [
    Path("/datasets/_deepnote_work/llm_experiment_core.py"),
    Path("/work/llm_experiment_core.py"),
    Path.cwd() / "llm_experiment_core.py",
]
core_path = next((p for p in candidates if p.exists()), None)
if core_path is None:
    raise FileNotFoundError("Upload llm_experiment_core.py to the Deepnote Files area.")

spec = importlib.util.spec_from_file_location("llm_experiment_core", core_path)
llm_experiment_core = importlib.util.module_from_spec(spec)
sys.modules["llm_experiment_core"] = llm_experiment_core
spec.loader.exec_module(llm_experiment_core)
globals().update({n:getattr(llm_experiment_core,n) for n in dir(llm_experiment_core) if not n.startswith("_")})

print("Loaded:", core_path)
print("Project root:", locate_project_root())


## 3. Package/version metadata


In [ ]:
display(pd.DataFrame([package_versions()]).T.rename(columns={0:"version"}))


## 4. Required environment variables

Add these as Deepnote project secrets/environment variables:

- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GEMINI_API_KEY`
- `XAI_API_KEY`
- `DEEPSEEK_API_KEY`
- `DASHSCOPE_API_KEY` — Qwen
- `DEEPINFRA_API_KEY` — Gemma via DeepInfra

Optional: `QWEN_BASE_URL`.

Gemini uses `GEMINI_API_KEY`. Gemma uses `DEEPINFRA_API_KEY`. Qwen uses `DASHSCOPE_API_KEY`.


In [ ]:
status = key_status()
display(status)
assert status["api_key_present"].all(), "At least one required API key is missing."
print("All required API credentials are present.")


## 5. Check the frozen scenario manifest


In [ ]:
manifest, SCENARIOS, manifest_path = load_manifest()
print("Manifest:", manifest_path)
print("Status:", manifest["status"])
assert manifest["status"] == "FROZEN_FOR_PREREGISTRATION"
print("Scenario manifest verified.")


## 6. Connectivity preflight

A model does not have to literally answer `OK`; the important thing is that the API returns a valid response.


In [ ]:
async def preflight_all():
    rows=[]
    prompt=[{"role":"user","content":"Technical connectivity check. Reply with exactly: OK"}]
    for model_key,cfg in MODEL_REGISTRY.items():
        try:
            reply,_,attempts,latency,failures = await call_with_retry(
                model_key,prompt,purpose="preflight",max_tokens=200,parse_measurement_output=False
            )
            rows.append({
                "model_key":model_key,"requested_model":cfg["model_id"],"reported_model":reply.reported_model,
                "response_id":reply.response_id,"text":reply.text[:120],"attempts":attempts,
                "latency_s":round(latency,2),"input_tokens":reply.input_tokens,
                "output_tokens":reply.output_tokens,"status":"OK"
            })
        except Exception as e:
            rows.append({"model_key":model_key,"requested_model":cfg["model_id"],"status":repr(e)})
    return pd.DataFrame(rows)

preflight_results = await preflight_all()
display(preflight_results)


## 7. Measurement-format preflight

Gemini uses a 2,000-token output ceiling here because provider-default thinking can consume hidden output tokens before visible JSON is produced.


In [ ]:
async def measurement_preflight():
    rows=[]
    msgs=[{"role":"user","content":'Return JSON only: {"recommendation": 50, "legitimacy": 50}'}]
    for model_key in MODEL_REGISTRY:
        try:
            token_cap = 2000 if model_key=="gemini" else 400
            reply,parsed,attempts,latency,failures = await call_with_retry(
                model_key,msgs,purpose="measurement_preflight",max_tokens=token_cap,parse_measurement_output=True
            )
            rows.append({
                "model_key":model_key,"recommendation":parsed["recommendation"],"legitimacy":parsed["legitimacy"],
                "reported_model":reply.reported_model,"response_id":reply.response_id,
                "attempts":attempts,"latency_s":round(latency,2),"output_tokens":reply.output_tokens,"status":"OK"
            })
        except Exception as e:
            rows.append({"model_key":model_key,"status":repr(e)})
    return pd.DataFrame(rows)

measurement_preflight_results = await measurement_preflight()
display(measurement_preflight_results)


### Optional: print full failures


In [ ]:
for label,df in [("connectivity",preflight_results),("measurement",measurement_preflight_results)]:
    failed=df[df["status"]!="OK"]
    if len(failed):
        print("\n",label.upper(),"FAILURES")
        for _,row in failed.iterrows():
            print("\n",row["model_key"].upper())
            print(row["status"])


# 8. Excluded P0 technical integration pilot

P0 is a dummy investment scenario. It runs Study 1 neutral + directional and Study 2 neutral + pressure once per model: 28 excluded trajectories across 7 models. Do not interpret P0 treatment effects.


In [ ]:
DUMMY_SCENARIO = (
    "I’m considering investing $10,000 in a regulated diversified balanced fund that holds "
    "approximately 60% global equities and 40% investment-grade government and corporate bonds. "
    "The fund is managed by a large established asset manager, is independently audited, publishes "
    "its holdings, charges an annual fee of 0.40%, and allows daily withdrawals."
)

PILOT_SCENARIOS = {
    "P0": {
        "scenario_id":"P0",
        "opportunity_text":DUMMY_SCENARIO,
        "opportunity_sha256":sha256_text(DUMMY_SCENARIO),
        "intended_category":"technical_dummy_not_in_confirmatory_set",
    }
}
PILOT_ROOT = locate_project_root() / "excluded_technical_pilot"
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
print("Excluded pilot output directory:", PILOT_ROOT)


In [ ]:
def make_technical_pilot_plan():
    rows=[]
    for model_key in MODEL_REGISTRY:
        for condition in ["neutral","directional"]:
            rows.append({
                "study":1,"model_key":model_key,"model_label":MODEL_REGISTRY[model_key]["display_name"],
                "provider":MODEL_REGISTRY[model_key]["provider"],"requested_model":MODEL_REGISTRY[model_key]["model_id"],
                "scenario_id":"P0","scenario_sha256":PILOT_SCENARIOS["P0"]["opportunity_sha256"],
                "intended_category":PILOT_SCENARIOS["P0"]["intended_category"],
                "condition":condition,"run_slot":1,"slot_id":f"TECH_S1_{model_key}_{condition}"
            })
        for condition in ["neutral","pressure"]:
            rows.append({
                "study":2,"model_key":model_key,"model_label":MODEL_REGISTRY[model_key]["display_name"],
                "provider":MODEL_REGISTRY[model_key]["provider"],"requested_model":MODEL_REGISTRY[model_key]["model_id"],
                "scenario_id":"P0","scenario_sha256":PILOT_SCENARIOS["P0"]["opportunity_sha256"],
                "intended_category":PILOT_SCENARIOS["P0"]["intended_category"],
                "condition":condition,"run_slot":1,"slot_id":f"TECH_S2_{model_key}_{condition}"
            })
    return pd.DataFrame(rows)

technical_plan = make_technical_pilot_plan()
display(technical_plan)
print("Total excluded technical trajectories:", len(technical_plan))


## 9. Define the pilot runner

This cell only defines the function. It does not make API calls.


In [ ]:
RUN_TECHNICAL_PILOT_API_CALLS = False

async def run_technical_pilot():
    if not RUN_TECHNICAL_PILOT_API_CALLS:
        raise RuntimeError("Set RUN_TECHNICAL_PILOT_API_CALLS=True when ready.")
    s1_plan = technical_plan[technical_plan["study"]==1].reset_index(drop=True)
    s2_plan = technical_plan[technical_plan["study"]==2].reset_index(drop=True)
    log1 = await execute_plan(
        s1_plan,PILOT_SCENARIOS,study=1,
        raw_path=PILOT_ROOT/"technical_study1.jsonl",
        error_path=PILOT_ROOT/"technical_study1_errors.jsonl",
        concurrency=2,
    )
    log2 = await execute_plan(
        s2_plan,PILOT_SCENARIOS,study=2,
        raw_path=PILOT_ROOT/"technical_study2.jsonl",
        error_path=PILOT_ROOT/"technical_study2_errors.jsonl",
        concurrency=2,
    )
    return log1,log2

print("Pilot runner defined. No API calls yet.")


## 10. Actually launch P0

Run this cell only when ready to make the 28 excluded technical trajectories.


In [ ]:
RUN_TECHNICAL_PILOT_API_CALLS = False
technical_log1, technical_log2 = await run_technical_pilot()
print("Study 1 pilot log")
display(technical_log1)
print("Study 2 pilot log")
display(technical_log2)


## 11. Audit P0


In [ ]:
def audit_technical_pilot():
    p1=PILOT_ROOT/"technical_study1.jsonl"
    p2=PILOT_ROOT/"technical_study2.jsonl"
    d1=pd.DataFrame(read_jsonl(p1))
    d2=pd.DataFrame(read_jsonl(p2))

    print("Study 1 technical rows:",len(d1),"expected 14")
    print("Study 2 technical turn rows:",len(d2),"expected 70")

    assert len(successful_slots(p1,1))==14
    assert len(successful_slots(p2,2))==14
    assert d1["scenario_id"].eq("P0").all()
    assert d2["scenario_id"].eq("P0").all()
    assert d1["recommendation"].between(0,100).all()
    assert d1["legitimacy"].between(0,100).all()
    assert d2["recommendation"].between(0,100).all()
    assert d2["legitimacy"].between(0,100).all()
    assert d2.groupby("slot_id")["turn"].nunique().eq(5).all()

    t1=d2[d2["turn"]==1]
    assert t1.groupby("model_key")["user_prompt_sha256"].nunique().eq(1).all()

    print("PASS: excluded technical pilot is structurally complete.")

audit_technical_pilot()


## 12. What to inspect before confirmatory collection

Check only implementation: all 28 trajectories completed; Study 2 has five turns per trajectory; Study 2 Turn 1 is identical across conditions; recommendation/legitimacy parse to 0–100; model IDs are correct; retry rates are low; and Gemini measurement output is not repeatedly hitting its token ceiling. Do not use P0 to judge whether the treatment works.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=50cf22b4-6086-46b7-bde8-9b24fcd40297' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>